In [2]:
# ===============================
# import libraries
# ===============================
import numpy as np
import pandas as pd
from pathlib import Path
import chartgazing as cg

cg.set_size("4:5")

In [3]:
# ===============================
# import data
# ===============================

# set data directory
DATA_DIR = Path("data/sets")

# combine every saved set
pokemon_all = pd.concat(
    [pd.read_csv(path, dtype={"number": str, "product_id": str, "variant": str}, parse_dates=["date"]) for path in DATA_DIR.glob("*.csv")],
    ignore_index=True
)

# additional data on each set
sets_df = pd.read_csv("data/sets_info.csv", parse_dates=["release_date"])
pokemon_all = pokemon_all.merge(sets_df, on="set_slug", how="left")

In [4]:
# ===============================
# clean data
# ===============================
pokemon_all["variant"] = pokemon_all["variant"].fillna("Standard")
pokemon_all["price"] = pokemon_all["price"].replace(0, np.nan)
pokemon_all["date"] = pokemon_all["date"].dt.to_period("M").dt.to_timestamp()
pokemon_all["decade"] = (pokemon_all["release_date"].dt.year // 10) * 10


for col in ["grade", "set_slug", "set_name", "language", "name", "variant"]:
    pokemon_all[col] = pokemon_all[col].astype("category")

In [ ]:
# ===============================
# Changes for all cards
# ===============================
GRADE = "Ungraded"
IMAGE_DIR = "images/4x5"
source = "Source: PriceCharting.com"

all_prices = pokemon_all.dropna(subset=["price"])
priced_cards = all_prices[all_prices["grade"] == GRADE]
latest_date = priced_cards["date"].max()
latest_cards = priced_cards[priced_cards["date"] == latest_date]

def fmt(value):
    return f"${value:,.0f}" if value >= 1000 else f"${value:.2f}"

def short_set(set_name):
    return set_name.removeprefix("Pokemon ")

def one_per_card(cards):
    return(
        cards.assign(not_standard=cards["variant"] != "Standard")
        .sort_values(["not_standard", "price"])
        .drop_duplicates(["set_slug", "number", "name"])
    )

,grade,date,price,set_slug,product_id,name,variant,number,set_name,language,release_date,promo,decade
15,Grade 9.5,2023-09-01,39.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990
16,Grade 9.5,2023-10-01,39.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990
17,Grade 9.5,2023-11-01,39.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990
18,Grade 9.5,2023-12-01,39.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990
19,Grade 9.5,2024-01-01,39.00,pokemon-1999-topps-movie,3560849,pikachu's vacation,Standard,42,Pokemon 1999 Topps Movie,English,1999-06-01,False,1990
...,...,...,...,...,...,...,...,...,...,...,...,...,...
13124125,Ungraded,2026-06-01,0.52,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False,2010
13124126,Ungraded,2026-07-01,0.70,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False,2010
13124127,Ungraded,2026-08-01,0.66,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False,2010
13124128,Ungraded,2026-09-01,0.64,pokemon-xy,806728,bunnelby,Reverse Holo,111,Pokemon XY,English,2014-02-05,False,2010


In [43]:
cards_2026 = all_prices[(all_prices["date"] >= "2026-01-01") & (all_prices["grade"] == GRADE)]

# cards_2026 = all_prices[all_prices["date"] >= "2026-01-01"]

year_start = cards_2026.loc[cards_2026["date"] == "2026-01-01", "product_id"].unique()

print(cards_2026.shape)

cards_2026 = cards_2026[cards_2026["product_id"].isin(year_start)]

cards_26 = cards_2026.groupby("date", as_index=False).agg(
    unique_cards = ("product_id", "nunique"),
    mean_price = ("price", "mean")
)

# cards_26["grade"] = cards_26["mean_price"].pct_change() * 100
cards_26["absolute_chng"] = cards_26["mean_price"] - cards_26["mean_price"].iloc[0]

cards_26["pct_chng"] = ((cards_26["mean_price"] - cards_26["mean_price"].iloc[0]) / cards_26["mean_price"]) * 100


cards_26

(394874, 13)


,date,unique_cards,mean_price,absolute_chng,pct_chng
0,2026-01-01,38445,17.343589,0.000000,0.000000
1,2026-02-01,38444,17.705800,0.362211,2.045721
2,2026-03-01,38444,18.400892,1.057303,5.745936
3,2026-04-01,38441,19.262375,1.918786,9.961314
4,2026-05-01,38442,20.528885,3.185296,15.516168
5,2026-06-01,38442,21.498796,4.155207,19.327625
6,2026-07-01,38442,22.286825,4.943236,22.180081
7,2026-08-01,38443,23.442201,6.098612,26.015528
8,2026-09-01,38442,24.478734,7.135145,29.148341
9,2026-10-01,38441,25.859566,8.515977,32.931632


In [ ]:
pokemon_all.info()



<class 'pandas.DataFrame'>
RangeIndex: 13124130 entries, 0 to 13124129
Data columns (total 13 columns):
 #   Column        Dtype         
---  ------        -----         
 0   grade         category      
 1   date          datetime64[us]
 2   price         float64       
 3   set_slug      category      
 4   product_id    str           
 5   name          category      
 6   variant       category      
 7   number        str           
 8   set_name      category      
 9   language      category      
 10  release_date  datetime64[us]
 11  promo         bool          
 12  decade        int32         
dtypes: bool(1), category(6), datetime64[us](2), float64(1), int32(1), str(2)
memory usage: 797.3 MB


In [ ]:
cards = pokemon_all.drop_duplicates("product_id")
cards["name"].astype(str).sample(40, random_state=2).to_list()

In [ ]:
GRADE_ORDER = ['Ungraded', 'Grade 7', 'Grade 8', 'Grade 9', 'Grade 9.5', 'PSA 10']



['Grade 9.5', 'Grade 7', 'Grade 9', 'PSA 10', 'Grade 8', 'Ungraded']
Categories (6, str): ['Grade 7', 'Grade 8', 'Grade 9', 'Grade 9.5', 'PSA 10', 'Ungraded']